# AQI Predictor — Exploratory Data Analysis

This notebook loads the feature dataset from the Hopsworks Feature Store and performs a comprehensive EDA.

**Contents:**
1. Data loading & overview
2. Descriptive statistics
3. AQI distribution (histogram + KDE)
4. AQI over time
5. Correlation heatmap
6. AQI by hour of day (box plot)
7. AQI by month (box plot)
8. Missing value heatmap
9. Pollutant–AQI scatter plots
10. Key findings


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Add project root to path
ROOT = Path('.').resolve().parent
sys.path.insert(0, str(ROOT))

import config

# Plotting style
plt.rcParams.update({
    'figure.facecolor': '#0f172a',
    'axes.facecolor': '#1e293b',
    'axes.edgecolor': '#334155',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'text.color': '#e2e8f0',
    'grid.color': '#1e3a5f',
    'grid.linewidth': 0.5,
    'font.family': 'sans-serif',
})
PALETTE = sns.color_palette('viridis', 12)

print('Imports OK')

## 1. Load Data from Hopsworks Feature Store

In [ ]:
import hopsworks

project = hopsworks.login(
    project=config.HOPSWORKS_PROJECT_NAME,
    api_key_value=config.HOPSWORKS_API_KEY,
)

fs = project.get_feature_store()
fg = fs.get_feature_group(name=config.FEATURE_GROUP_NAME, version=config.FEATURE_GROUP_VERSION)

df = fg.read()
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
df = df.sort_values('timestamp').reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(f'Date range:    {df["timestamp"].min()} → {df["timestamp"].max()}')

## 2. Descriptive Statistics

In [ ]:
print('=== DataFrame Info ===')
df.info()
print()
print('=== Descriptive Statistics ===')
df.describe().T.style.background_gradient(cmap='viridis', axis=1)

## 3. AQI Distribution (Histogram + KDE)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('AQI Distribution', fontsize=14, y=1.01)

# Histogram
axes[0].hist(df['aqi'].dropna(), bins=50, color='#7c3aed', edgecolor='#0f172a', alpha=0.85)
axes[0].set_title('Histogram')
axes[0].set_xlabel('AQI')
axes[0].set_ylabel('Count')
axes[0].grid(alpha=0.3)

# AQI threshold lines
for threshold, label, color in [
    (50, 'Good', '#22c55e'),
    (100, 'Moderate', '#eab308'),
    (150, 'Sensitive', '#f97316'),
    (200, 'Unhealthy', '#ef4444'),
]:
    axes[0].axvline(threshold, color=color, linestyle='--', linewidth=1.2, label=label)
axes[0].legend(fontsize=8)

# KDE
aqi_clean = df['aqi'].dropna()
from scipy.stats import gaussian_kde
x_range = np.linspace(aqi_clean.min(), aqi_clean.max(), 300)
kde = gaussian_kde(aqi_clean)
axes[1].fill_between(x_range, kde(x_range), alpha=0.6, color='#38bdf8')
axes[1].plot(x_range, kde(x_range), color='#0ea5e9', linewidth=2)
axes[1].set_title('KDE (Kernel Density Estimate)')
axes[1].set_xlabel('AQI')
axes[1].set_ylabel('Density')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(config.PLOTS_DIR / 'eda_aqi_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. AQI Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(df['timestamp'], df['aqi'], color='#60a5fa', linewidth=0.8, alpha=0.9, label='AQI')
ax.fill_between(df['timestamp'], df['aqi'], alpha=0.15, color='#3b82f6')

# Rolling 24h mean
rolling = df.set_index('timestamp')['aqi'].rolling('24h').mean()
ax.plot(rolling.index, rolling.values, color='#f59e0b', linewidth=1.5, label='24h Rolling Mean')

ax.axhline(150, color='#ef4444', linestyle='--', linewidth=1, label='Unhealthy (150)')
ax.set_title('AQI Over Time', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('AQI')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(config.PLOTS_DIR / 'eda_aqi_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation Heatmap

In [ ]:
from src.feature_engineer import FEATURE_COLS, TARGET_COLS

numeric_cols = [c for c in FEATURE_COLS + TARGET_COLS if c in df.columns]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(18, 16))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=False,
    cmap='coolwarm', center=0, square=True,
    linewidths=0.3, linecolor='#0f172a',
    ax=ax, cbar_kws={'shrink': 0.7},
)
ax.set_title('Feature Correlation Matrix', fontsize=14, pad=16)
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.tick_params(axis='y', rotation=0, labelsize=7)

plt.tight_layout()
plt.savefig(config.PLOTS_DIR / 'eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. AQI by Hour of Day (Box Plot)

In [ ]:
df['hour'] = df['timestamp'].dt.hour

fig, ax = plt.subplots(figsize=(16, 6))
hourly_data = [df[df['hour'] == h]['aqi'].dropna().values for h in range(24)]

bp = ax.boxplot(
    hourly_data, patch_artist=True,
    medianprops=dict(color='#f59e0b', linewidth=2),
    flierprops=dict(marker='o', markersize=2, alpha=0.3, color='#60a5fa'),
)

colors = plt.cm.viridis(np.linspace(0.2, 0.9, 24))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_xticks(range(1, 25))
ax.set_xticklabels([f'{h:02d}:00' for h in range(24)], rotation=45, fontsize=8)
ax.set_title('AQI Distribution by Hour of Day', fontsize=14)
ax.set_xlabel('Hour (UTC)')
ax.set_ylabel('AQI')
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(config.PLOTS_DIR / 'eda_aqi_by_hour.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. AQI by Month (Box Plot)

In [ ]:
import calendar

df['month'] = df['timestamp'].dt.month

fig, ax = plt.subplots(figsize=(14, 6))
months_present = sorted(df['month'].unique())
monthly_data = [df[df['month'] == m]['aqi'].dropna().values for m in months_present]

bp = ax.boxplot(
    monthly_data, patch_artist=True,
    medianprops=dict(color='#f59e0b', linewidth=2),
    flierprops=dict(marker='o', markersize=2, alpha=0.3, color='#60a5fa'),
)

colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(months_present)))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_xticks(range(1, len(months_present) + 1))
ax.set_xticklabels([calendar.month_abbr[m] for m in months_present])
ax.set_title('AQI Distribution by Month', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('AQI')
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(config.PLOTS_DIR / 'eda_aqi_by_month.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Missing Value Heatmap

In [ ]:
from src.feature_engineer import FEATURE_COLS

cols_to_check = [c for c in FEATURE_COLS if c in df.columns]
missing = df[cols_to_check].isnull()

fig, ax = plt.subplots(figsize=(16, max(5, len(cols_to_check) * 0.35)))
sns.heatmap(
    missing.T,
    cmap=['#1e3a5f', '#ef4444'],
    cbar=False,
    yticklabels=True,
    xticklabels=False,
    ax=ax,
)
ax.set_title('Missing Values Heatmap (red = missing)', fontsize=13)
ax.set_xlabel('Samples →')
ax.tick_params(axis='y', labelsize=8)

# Print summary
miss_pct = (missing.sum() / len(df) * 100).sort_values(ascending=False)
print('Missing value summary:')
print(miss_pct[miss_pct > 0].to_string())

plt.tight_layout()
plt.savefig(config.PLOTS_DIR / 'eda_missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Pollutant Correlation with AQI (Scatter Plots)

In [ ]:
pollutants = ['pm25', 'pm10', 'o3', 'no2', 'so2', 'co']
available_pollutants = [p for p in pollutants if p in df.columns]

n_cols = 3
n_rows = int(np.ceil(len(available_pollutants) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4.5))
axes = axes.flatten()

colors_poll = plt.cm.viridis(np.linspace(0.1, 0.9, len(available_pollutants)))

for i, (pollutant, color) in enumerate(zip(available_pollutants, colors_poll)):
    ax = axes[i]
    valid = df[['aqi', pollutant]].dropna()
    
    # Scatter with transparency
    ax.scatter(valid[pollutant], valid['aqi'], alpha=0.25, s=8, color=color)
    
    # Best-fit line
    if len(valid) > 1:
        m, b = np.polyfit(valid[pollutant], valid['aqi'], 1)
        x_line = np.linspace(valid[pollutant].min(), valid[pollutant].max(), 100)
        ax.plot(x_line, m * x_line + b, color='#f59e0b', linewidth=2, label=f'r={valid.corr()["aqi"][pollutant]:.2f}')
        ax.legend(fontsize=9)
    
    ax.set_xlabel(pollutant.upper())
    ax.set_ylabel('AQI')
    ax.set_title(f'AQI vs {pollutant.upper()}')
    ax.grid(alpha=0.3)

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Pollutant vs AQI Scatter Plots', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(config.PLOTS_DIR / 'eda_pollutant_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Key Findings & Conclusions

### Summary

Based on the exploratory analysis of the AQI feature dataset, the following key insights emerge:

1. **AQI Distribution**: The data exhibits a right-skewed distribution, with the majority of readings falling in the *Moderate* (51–100) range. Extreme *Hazardous* events (>300) are rare but present — important tail events for model robustness.

2. **Temporal Patterns**:
   - AQI peaks during **early morning hours** (06:00–09:00) and **evening rush hours** (17:00–21:00), consistent with traffic emissions.
   - **Winter months** (Nov–Feb) show significantly higher AQI due to temperature inversion, reduced dispersion, and heating emissions.

3. **Top Correlated Pollutants**:
   - **PM2.5** has the strongest positive correlation with overall AQI, followed by **PM10**.
   - **O3** (ozone) shows a distinct diurnal cycle with a peak in the afternoon.
   - **CO** and **NO2** correlate with traffic intensity.

4. **Missing Values**: Pollutants such as **SO2** and **CO** have the highest missingness rates (~10–20%). The median imputation strategy used in the training pipeline is appropriate for these cases.

5. **Lag Features**: The strong auto-correlation of AQI (visible in the time-series plot) justifies using `aqi_lag_1h` and `rolling_aqi_mean_3h` as strong predictive signals for short-horizon forecasts.

6. **Feature Engineering Effectiveness**: Cyclic encodings of hour and month effectively capture periodic patterns without introducing discontinuities at boundaries.

### Recommendations
- Collect **more winter data** to improve model accuracy during high-AQI periods.
- Consider adding **wind-direction** binning to capture directional pollution effects.
- Explore **station-level granularity** if multiple WAQI stations are available within the city.
